In [ ]:
!git clone https://github.com/autonlab/auton-survival.git
%cd auton-survival
!pip install -r requirements.txt
!pip install .
%cd ..

In [23]:
# --- Monkeypatch scipy.integrate.trapz for lifelines/auton_survival ---

import numpy as _np
import scipy.integrate as _si

if not hasattr(_si, "trapz"):
    def _trapz(y, x=None, dx=1.0, axis=-1):
        return _np.trapz(y, x=x, dx=dx, axis=axis)
    _si.trapz = _trapz

print("scipy.integrate.trapz patched:", hasattr(_si, "trapz"))


scipy.integrate.trapz patched: True


In [24]:
import pandas as pd

df = pd.read_csv('cleaned_data.csv')

clean_feature_cols_asd = [

    # -----------------
    # Demographics
    # -----------------
    "Sex",
    "Age",
    "BMI",
    "prior back surgeries? (y=1)",

    # -----------------
    # Pre-op diagnosis flags (from Pre op Diagnosis text)
    # -----------------
    "dx_adjacent_segment",
    "dx_spondylolisthesis",
    "dx_spondylosis",
    "dx_stenosis",
    "dx_scoliosis",
    "dx_flat_back",
    "dx_sagittal_imbalance",
    "dx_post_laminectomy",
    "dx_deformity",

    # -----------------
    # Fusion levels (index constructs)
    # -----------------
    "T12-L1",
    "L1-L2",
    "L2-L3",
    "L3-L4",
    "L4-L5",
    "L5-S1",

    # Construct summary features you engineered
    "levels_fused_count",
    "construct_span_levels",
    "thoracolumbar_junction",
    "upper_lumbar",
    "lower_lumbar",
    "lumbosacral",

    # -----------------
    # Surgical approach / technique
    # -----------------
    "LLIF?",                # whether lateral/LLIF was done at all
    "Perc screws?",
    "Open",
    "Open Check V2",
    "Standalone XLIF Check",
    "Retroperitoneal Approach (LLIF ± ALIF)",
    "Anterior + Posterior Apporoach",
    "Osteotomies (yes/no)",
    "osteotomy level",
    "ALIF Count",
    "Lateral Count",
    "ACR (y=1)",
    "ACR level",

    # Higher-level surgery flags
    "revision_surgery",
    "deformity_case_text",
    "llif_or_lateral_text",
    "xlif_text",
    "alif_text",

    # -----------------
    # Preoperative spinopelvic parameters
    # -----------------
    "Average PI",
    "PI-LL angle mismatch",
    "ABS PI-LL angle mismatch",
    "PI-LL Mismatch Category (1 = mismatch > +/- 9",
    "PI-LL Mismatch Category (1 = mismatch > +/- 10",
    "(1 = PI>50)",   # this matches your full column list

    # -----------------
    # Immediate postoperative alignment (non-leaky)
    # -----------------
    "Post-op SS",
    "post PI",
    "post PT",
    "post LL",
    "post SVA",

    # -----------------
    # Immediate postop complications (index hospitalization)
    # -----------------
    "infection 1=yes",
    "DVT  1=yes",
    "PE  1=yes",
    "MI 1=yes",
    "femoral palsy (knee extension weakness) 1=yes",
    "hip flexion weakness (iliopsoas weakness)  1=yes",
    "acute thigh paresthesia (immediate post op)",
    "psoas hematoma",

    # -----------------
    # Hospital course
    # -----------------
    "length of hospital stay (d)",
]

import pandas as pd


# 2. Clean column names a bit
df.columns = (
    df.columns
      .str.strip()
      .str.replace("\n", " ", regex=False)
)


target = "Time Until ASD Diagnosis (months)"

# Keep only rows where the target is known (patients who actually developed ASD)
asd_df = df[df[target].notna()].copy()
asd_df.shape

event_col = "REVERIFIED ASD"
time_asd_col = "Time Until ASD Diagnosis (months)"
time_no_asd_col = "Time Without_ASD (months)"   # used ONLY for label, not as feature

# df is assumed to be your cleaned dataframe (e.g. df_lace or df_clean)

df_surv = df.copy()

# event: 1 if ASD, else 0
df_surv[event_col] = df_surv[event_col].fillna(0).astype(int)

# time: ASD time if event==1, otherwise Time Without_ASD
df_surv["time_surv"] = np.where(
    df_surv[event_col] == 1,
    df_surv[time_asd_col],
    df_surv[time_no_asd_col],
)

# Drop rows with missing survival time
df_surv = df_surv.dropna(subset=["time_surv"])

# ---------------------------------
# 2. Feature preprocessing
# ---------------------------------

X = df_surv[clean_feature_cols_asd].copy()

# 2a. Coerce numeric-like columns
numeric_like_cols = [
    "BMI",
    "ALIF Count",
    "Lateral Count",
    "Average PI",
    "PI-LL angle mismatch",
    "ABS PI-LL angle mismatch",
    "Post-op SS",
    "post PI",
    "post PT",
    "post LL",
    "post SVA",
    "length of hospital stay (d)",
]

for col in numeric_like_cols:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

# 2b. One-hot encode categorical columns
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Simple imputation: fill remaining NaNs
X_encoded = X_encoded.fillna(0)

# Targets
durations_all = df_surv["time_surv"].astype(float).values
events_all = df_surv[event_col].astype(bool).values




In [25]:
import pandas as pd
import numpy as np

from auton_survival.experiments import SurvivalRegressionCV

# ---------------------------
# 1. Build survival labels (same logic as before)
# ---------------------------

event_col = "REVERIFIED ASD"
time_asd_col = "Time Until ASD Diagnosis (months)"
time_no_asd_col = "Time Without_ASD (months)"   # ONLY for label, not a feature

df_surv = df.copy()

# event: 1 if ASD, else 0
df_surv[event_col] = df_surv[event_col].fillna(0).astype(int)

# time: ASD time if event==1, otherwise Time Without_ASD
df_surv["time_surv"] = np.where(
    df_surv[event_col] == 1,
    df_surv[time_asd_col],
    df_surv[time_no_asd_col],
)

# Drop rows with missing survival time
df_surv = df_surv.dropna(subset=["time_surv"]).copy()

print("Full cohort survival rows:", df_surv.shape[0])
print("Events (ASD):", int(df_surv[event_col].sum()))
print("Non-events:", int((df_surv[event_col] == 0).sum()))

# ---------------------------
# 2. Features for auton_survival
# ---------------------------

# Use the structured ASD feature set we defined
features = df_surv[clean_feature_cols_asd].copy()

# auton_survival expects an "outcomes" object with columns 'time' and 'event'
outcomes = pd.DataFrame({
    "time": df_surv["time_surv"].astype(float).values,
    "event": df_surv[event_col].astype(int).values,
})

# Identify categorical vs numeric features
cat_feats = features.select_dtypes(include=["object"]).columns.tolist()
num_feats = [c for c in features.columns if c not in cat_feats]

print("Features shape:", features.shape)
print("Categorical features:", cat_feats)
print("Numeric-like features (first 10):", num_feats[:10])


Full cohort survival rows: 546
Events (ASD): 122
Non-events: 424
Features shape: (546, 63)
Categorical features: ['osteotomy level', 'ACR level']
Numeric-like features (first 10): ['Sex', 'Age', 'BMI', 'prior back surgeries? (y=1)', 'dx_adjacent_segment', 'dx_spondylolisthesis', 'dx_spondylosis', 'dx_stenosis', 'dx_scoliosis', 'dx_flat_back']


In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from auton_survival.estimators import SurvivalModel
from auton_survival.metrics import survival_regression_metric

# Use the same encoded features & outcomes
X_mat = X_encoded.values
y_out = outcomes.copy()  # columns: 'time', 'event'

horizons = np.array([12, 24, 36])  # months
n_splits = 5

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

cv_ctd_scores = []

fold_idx = 1
for train_idx, test_idx in skf.split(X_mat, y_out["event"].values):
    print(f"\n===== auton DeepCoxPH CV fold {fold_idx}/{n_splits} =====")
    fold_idx += 1

    X_train = X_mat[train_idx]
    X_test = X_mat[test_idx]

    y_train = y_out.iloc[train_idx].reset_index(drop=True)
    y_test = y_out.iloc[test_idx].reset_index(drop=True)

    # Build & fit auton SurvivalModel for DeepCoxPH
    # You can embed your favorite hyperparams here (from the grid you used)
    model = SurvivalModel(
        model="dcph",
        random_seed=42,
        layers=[64, 32],      # e.g., choose the best config from experiment
        learning_rate=5e-4,   # or 1e-3
        batch_size=128,
    )

    model.fit(pd.DataFrame(X_train, index=y_train.index),
              y_train)

    # Predict survival at horizons on the held-out fold
    pred_surv_test = model.predict_survival(
        pd.DataFrame(X_test, index=y_test.index),
        times=horizons
    )

    # Time-dependent C-index on this fold
    ctd_fold = survival_regression_metric(
        metric="ctd",
        outcomes=y_test,
        predictions=pred_surv_test,
        times=horizons,
        outcomes_train=y_train
    )

    # survival_regression_metric returns an array (per horizon); take mean over horizons
    ctd_mean_fold = float(np.mean(ctd_fold))
    cv_ctd_scores.append(ctd_mean_fold)

    print("Fold CTD (per horizon):", ctd_fold)
    print("Fold CTD (mean over horizons):", ctd_mean_fold)

cv_ctd_scores = np.array(cv_ctd_scores)
print("\n===== auton DeepCoxPH cross-validated C-index =====")
print("Mean CTD:", np.mean(cv_ctd_scores))
print("SD CTD:  ", np.std(cv_ctd_scores))



===== auton DeepCoxPH CV fold 1/5 =====


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:00<00:00, 146.79it/s]


Fold CTD (per horizon): [0.5214674456593634, 0.5041601567594229, 0.5041601567594229]
Fold CTD (mean over horizons): 0.5099292530594032

===== auton DeepCoxPH CV fold 2/5 =====


100%|██████████| 50/50 [00:00<00:00, 156.03it/s]


...

Fold CTD (per horizon): [0.5407730297772149, 0.5831392868743903, 0.5784451992955416]
Fold CTD (mean over horizons): 0.5674525053157157

===== auton DeepCoxPH CV fold 4/5 =====


 78%|███████▊  | 39/50 [00:00<00:00, 148.04it/s]


Fold CTD (per horizon): [0.6258940345484489, 0.6948033174020185, 0.6557101374009783]
Fold CTD (mean over horizons): 0.6588024964504818

===== auton DeepCoxPH CV fold 5/5 =====


100%|██████████| 50/50 [00:00<00:00, 156.94it/s]

Fold CTD (per horizon): [0.5328428571818127, 0.5422400290222738, 0.5537961454998847]
Fold CTD (mean over horizons): 0.542959677234657

===== auton DeepCoxPH cross-validated C-index =====
Mean CTD: 0.5741959419603875
SD CTD:   0.05024105560827476
